# 23. Project One: Tracking a Virus as It Evolves

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `choose.py`

Choose a sample of SARS-CoV-2 genomes, a few from each major lineage.


In [ ]:
"""Choose a sample of SARS-CoV-2 genomes, a few from each major lineage.

Asks NCBI Virus, through the NCBI Datasets API, for complete genomes of
each lineage and keeps ten with a full collection date. NCBI adds genomes
every week, so a run today chooses different ones from the book's; the
book's choice is in samples.tsv, which download.py reads.

uv run choose.py            writes my_samples.tsv
"""

import csv
import json
import random
import time
import urllib.error
import urllib.parse
import urllib.request

API = "https://api.ncbi.nlm.nih.gov/datasets/v2/virus/taxon/2697049"

# Pango lineages in the order they spread, with the name the World Health
# Organization gave the variants of concern.
LINEAGES = {
    "A": "",
    "B": "",
    "B.1": "",
    "B.1.1.7": "Alpha",
    "B.1.351": "Beta",
    "P.1": "Gamma",
    "B.1.617.2": "Delta",
    "BA.1": "Omicron",
    "BA.2": "Omicron",
    "BA.5.2.1": "Omicron",
    "XBB.1.5": "Omicron",
    "JN.1": "Omicron",
    "KP.3": "Omicron",
}
PER_LINEAGE = 10

# The service runs out of time on Alpha, which has hundreds of thousands
# of genomes, unless the question is narrowed to one country.
NARROW = {"B.1.1.7": "Nigeria"}
ATTEMPTS = 4  # the service fails now and then, and works on a second ask


def reports(lineage, how_many=200):
    filters = {
        "filter.pangolin_classification": lineage,
        "filter.complete_only": "true",
        "page_size": how_many,
    }
    if lineage in NARROW:
        filters["filter.geo_location"] = NARROW[lineage]
    query = urllib.parse.urlencode(filters)
    for attempt in range(1, ATTEMPTS + 1):
        try:
            with urllib.request.urlopen(f"{API}/dataset_report?{query}",
                                        timeout=300) as response:
                return json.load(response).get("reports", [])
        except urllib.error.HTTPError as problem:
            if problem.code < 500 or attempt == ATTEMPTS:
                raise
            print(f"  {lineage}: {problem.code} from the service, "
                  f"asking again ({attempt} of {ATTEMPTS - 1})")
            time.sleep(5 * attempt)


def main():
    random.seed(1)
    rows = []
    for lineage, who in LINEAGES.items():
        usable = []
        for r in reports(lineage):
            date = r.get("isolate", {}).get("collection_date", "")
            human = r.get("host", {}).get("tax_id") == 9606
            if human and len(date) == 10 and r.get("length", 0) >= 29_000:
                usable.append({
                    "accession": r["accession"],
                    "lineage": lineage,
                    "who": who,
                    "date": date,
                    "country": r.get("location", {})
                    .get("geographic_location", "").split(":")[0],
                })
        chosen = random.sample(usable, min(PER_LINEAGE, len(usable)))
        print(f"{lineage:10} {len(usable):4} usable, "
              f"{len(chosen)} chosen")
        rows += sorted(chosen, key=lambda row: row["date"])
    with open("my_samples.tsv", "w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=rows[0], delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)


if __name__ == "__main__":
    main()

## `clean.py`

Keep only genomes that are long enough and have few unknown bases.


In [ ]:
"""Keep only genomes that are long enough and have few unknown bases.

The thresholds are arguments, with the defaults the book used, so that the
pipeline can pass the values in config.yaml and you can try others without
editing this file.

uv run clean.py                    reads genomes.fasta, writes clean.fasta
uv run clean.py --max-unknown 0.05  a looser threshold
"""

import argparse
import csv

from Bio import SeqIO


def unknown_share(sequence):
    """The share of positions that are not a plain A, C, G or T.

    N is the commonest, but a consensus may also hold an IUPAC letter
    such as Y for "C or T", and those are not called bases either.
    """
    called = sum(sequence.count(base) for base in "ACGT")
    return 1 - called / len(sequence)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--min-length", type=int, default=29_000,
                        help="shortest genome to keep (the reference is "
                             "29,903 bases)")
    parser.add_argument("--max-unknown", type=float, default=0.01,
                        help="largest share of uncalled bases to allow")
    args = parser.parse_args()

    with open("samples.tsv") as f:
        lineage = {row["accession"]: row["lineage"]
                   for row in csv.DictReader(f, delimiter="\t")}

    kept, dropped = [], []
    for record in SeqIO.parse("genomes.fasta", "fasta"):
        sequence = str(record.seq).upper()
        unknown = unknown_share(sequence)
        if (len(sequence) >= args.min_length
                and unknown <= args.max_unknown):
            record.description = ""
            kept.append(record)
        else:
            dropped.append((record.id, lineage[record.id], len(sequence),
                            unknown))

    SeqIO.write(kept, "clean.fasta", "fasta")
    print(f"kept {len(kept)} of {len(kept) + len(dropped)} genomes, "
          f"at most {args.max_unknown:.0%} unknown")
    for accession, name, length, unknown in dropped:
        print(f"  dropped {accession:11} {name:10} {length:6,} bases, "
              f"{unknown:.1%} unknown")


if __name__ == "__main__":
    main()

## `mutations.py`

List every difference between each genome and the reference.


In [ ]:
"""List every difference between each genome and the reference.

Reads aligned.fasta, where MAFFT kept the reference's numbering, so a
column of the alignment is a position in NC_045512.2. Writes one row per
mutation per genome, with its amino acid change where it falls in a gene.

uv run mutations.py        writes mutations.tsv
"""

import csv
from collections import Counter

from Bio import SeqIO
from Bio.Seq import Seq

# A consensus sequence is not limited to N for an uncertain position. The
# IUPAC alphabet has a letter for each pair of bases, such as Y for "C or
# T", and a genome that uses them will quietly produce amino acid changes
# that were never called. Anything that is not one plain base is unknown.
BASES = "ACGT"


def coding_positions(record):
    """{gene: [reference position of every base of its CDS, in order]}."""
    genes = {}
    for feature in record.features:
        if feature.type != "CDS":
            continue
        name = feature.qualifiers["gene"][0]
        product = feature.qualifiers["product"][0]
        if name == "ORF1ab" and "ORF1a " in product:
            # ORF1a is the shorter form of the same reading frame
            continue
        positions = []
        for part in feature.location.parts:
            positions += list(range(int(part.start), int(part.end)))
        genes[name] = positions
    return genes


def peptides(record):
    """[(start, end, name)] for the mature proteins cut out of ORF1ab."""
    found, seen = [], set()
    for feature in record.features:
        if feature.type != "mat_peptide":
            continue
        name = feature.qualifiers["product"][0]
        if name in seen:
            continue
        seen.add(name)
        found.append((int(feature.location.start),
                      int(feature.location.end), name))
    return found


def covered(sequence):
    """The first and last position the sequencing actually reached.

    Almost no SARS-CoV-2 genome covers the extreme ends of the molecule,
    and MAFFT writes the missing ends as gaps. Counting those gaps as
    deletions would add hundreds of false mutations to every genome.
    """
    first = len(sequence) - len(sequence.lstrip("-"))
    last = len(sequence.rstrip("-")) - 1
    return first, last


def protein_changes(reference, sample, positions, span):
    """Compare one gene codon by codon. Yields (codon number, change)."""
    for index in range(0, len(positions) - 2, 3):
        codon = positions[index:index + 3]
        if not all(span[0] <= p <= span[1] for p in codon):
            continue
        theirs = "".join(sample[p] for p in codon)
        if any(base not in BASES for base in theirs):
            if theirs == "---":
                yield index // 3 + 1, "del"
            continue
        ours = "".join(reference[p] for p in codon)
        if ours == theirs:
            continue
        before = Seq(ours).translate()
        after = Seq(theirs).translate()
        if before != after:
            yield index // 3 + 1, f"{before}{index // 3 + 1}{after}"


def main():
    annotation = SeqIO.read("reference.gb", "genbank")
    genes = coding_positions(annotation)
    mature = peptides(annotation)
    gene_of = {}
    for name, positions in genes.items():
        for p in positions:
            gene_of.setdefault(p, name)

    records = {r.id: str(r.seq).upper()
               for r in SeqIO.parse("aligned.fasta", "fasta")}
    reference = records.pop(annotation.id)
    with open("samples.tsv") as f:
        samples = {row["accession"]: row
                   for row in csv.DictReader(f, delimiter="\t")}

    rows, counts = [], Counter()
    for accession, sequence in records.items():
        sample = samples[accession]
        span = covered(sequence)
        changes = {}
        for gene, positions in genes.items():
            for codon, change in protein_changes(reference, sequence,
                                                 positions, span):
                changes[gene, codon] = change
        pairs = enumerate(zip(reference, sequence))
        for position, (ours, theirs) in pairs:
            if not span[0] <= position <= span[1]:
                continue
            if ours == theirs or (theirs not in BASES and theirs != "-"):
                continue
            gene = gene_of.get(position, "")
            codon = 0
            if gene:
                codon = genes[gene].index(position) // 3 + 1
            rows.append({
                "accession": accession,
                "lineage": sample["lineage"],
                "date": sample["date"],
                "position": position + 1,  # one based, as papers write it
                "reference": ours,
                "sample": theirs,
                "gene": gene,
                "protein": changes.get((gene, codon), ""),
                "nsp": name_of_peptide(mature, position),
            })
            counts[accession] += 1

    rows.sort(key=lambda r: (r["lineage"], r["accession"], r["position"]))
    with open("mutations.tsv", "w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=rows[0], delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)
    print(f"{len(rows):,} differences in {len(records)} genomes")
    fewest = min(counts.values())
    most = max(counts.values())
    print(f"from {fewest} in one genome to {most} in another")


def name_of_peptide(mature, position):
    for start, end, name in mature:
        if start <= position < end:
            return name
    return ""


if __name__ == "__main__":
    main()

## `locate.py`

Where in the genome the mutations gather.


In [ ]:
"""Where in the genome the mutations gather.

Counts mutations per gene, and per thousand bases of gene, so that a long
gene is not called a hotspot merely for being long. Then counts how many
of the sampled genomes carry each spike mutation.

uv run locate.py           reads mutations.tsv, writes per_gene.tsv
"""

import csv
from collections import Counter, defaultdict

from Bio import SeqIO


def gene_lengths(record):
    lengths = {}
    for feature in record.features:
        if feature.type != "gene":
            continue
        name = feature.qualifiers["gene"][0]
        lengths[name] = len(feature.location)
    return lengths


def main():
    reference = SeqIO.read("reference.gb", "genbank")
    lengths = gene_lengths(reference)
    with open("mutations.tsv") as f:
        rows = list(csv.DictReader(f, delimiter="\t"))
    analyzed = sum(1 for _ in SeqIO.parse("clean.fasta", "fasta"))
    # two genomes differ from the reference nowhere at all, so they have
    # no rows in mutations.tsv; Chapter 23 asks why
    genomes = len({row["accession"] for row in rows})

    # one mutation per genome counts once; a position shared by fifty
    # genomes is one place the virus changed, not fifty
    places = defaultdict(set)
    for row in rows:
        places[row["gene"] or "between genes"].add(int(row["position"]))
    events = Counter(row["gene"] or "between genes" for row in rows)

    table = []
    for gene, distinct in places.items():
        length = lengths.get(gene, 0)
        table.append({
            "gene": gene,
            "length": length,
            "positions": len(distinct),
            "per_kb": (round(1000 * len(distinct) / length, 2)
                       if length else ""),
            "changes": events[gene],
        })
    table.sort(key=lambda row: row["per_kb"] or 0, reverse=True)
    with open("per_gene.tsv", "w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=table[0], delimiter="\t")
        writer.writeheader()
        writer.writerows(table)

    print(f"{len(rows):,} changes in {genomes} of the "
          f"{analyzed} genomes\n")
    print(f"{'gene':14} {'bases':>6} {'places':>7} {'per kb':>7}")
    for row in table:
        print(f"{row['gene']:14} {row['length'] or '':>6} "
              f"{row['positions']:>7} {row['per_kb'] or '':>7}")

    spike = Counter(row["protein"] for row in rows
                    if row["gene"] == "S" and row["protein"]
                    and row["protein"] != "del")
    print(f"\nthe spike changes most of the {analyzed} genomes share:")
    for change, carriers in spike.most_common(12):
        print(f"  {change:8} {carriers:3} genomes  "
              f"{carriers / analyzed:.0%}")


if __name__ == "__main__":
    main()

## `timeline.py`

When each change first appears, and how fast changes accumulate.


In [ ]:
"""When each change first appears, and how fast changes accumulate.

Every genome carries a collection date, so the sample is a record of the
virus through time. Two questions follow: how many changes a genome picked
up per year, and when each famous spike change was first collected.

uv run timeline.py         reads mutations.tsv, writes clock.tsv
"""

import csv
from collections import Counter, defaultdict
from datetime import date

from Bio import SeqIO

WATCH = ["D614G", "N501Y", "P681H", "P681R", "L452R", "T478K", "E484K",
         "K417N", "Q498R", "L455S"]
START = date(2019, 12, 24)  # the collection date of the reference genome


def read_days(text):
    year, month, day = (int(part) for part in text.split("-"))
    return (date(year, month, day) - START).days


def main():
    with open("mutations.tsv") as f:
        rows = list(csv.DictReader(f, delimiter="\t"))
    with open("samples.tsv") as f:
        samples = list(csv.DictReader(f, delimiter="\t"))
    # the genomes the analysis actually ran on, named by the cleaned file,
    # not by samples.tsv, which still lists the ones clean.py dropped
    analyzed = {record.id
                for record in SeqIO.parse("clean.fasta", "fasta")}

    clock = dated_counts(rows, samples, analyzed)
    with open("clock.tsv", "w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=clock[0], delimiter="\t")
        writer.writeheader()
        writer.writerows(clock)

    rate, intercept = fit([row["days"] for row in clock],
                          [row["substitutions"] for row in clock])
    days = [row["days"] for row in clock]
    print(f"{len(clock)} genomes, collected over "
          f"{max(days) - min(days)}"
          " days")
    print(f"substitutions per year: {rate * 365:.1f}")
    print("at the reference's own date the fit expects "
          f"{intercept:.1f}\n")

    first_seen(rows)


def dated_counts(rows, samples, analyzed):
    """One row per genome: date, age in days, substitution count."""
    substitutions = Counter()
    for row in rows:
        if row["sample"] != "-":
            substitutions[row["accession"]] += 1
    clock = []
    for sample in samples:
        accession = sample["accession"]
        if accession not in analyzed:
            continue  # clean.py dropped it
        clock.append({
            "accession": accession,
            "lineage": sample["lineage"],
            "date": sample["date"],
            "days": read_days(sample["date"]),
            "substitutions": substitutions.get(accession, 0),
        })
    return clock


def first_seen(rows):
    """The earliest collection date carrying each watched spike change."""
    first = defaultdict(list)
    for row in rows:
        if row["gene"] == "S" and row["protein"] in WATCH:
            first[row["protein"]].append((row["date"], row["lineage"]))
    print(f"{'change':8} {'first seen':12} {'in':10} {'genomes':>8}")
    for change in WATCH:
        seen = sorted(first[change])
        if not seen:
            continue
        print(f"{change:8} {seen[0][0]:12} {seen[0][1]:10} "
              f"{len(seen):>8}")


def fit(x, y):
    """Least squares slope and intercept, written out in full."""
    n = len(x)
    mean_x, mean_y = sum(x) / n, sum(y) / n
    top = sum((a - mean_x) * (b - mean_y) for a, b in zip(x, y))
    bottom = sum((a - mean_x) ** 2 for a in x)
    slope = top / bottom
    return slope, mean_y - slope * mean_x


if __name__ == "__main__":
    main()

## `tree.py`

A tree of the sampled genomes, and a check that the lineages hold up.


In [ ]:
"""A tree of the sampled genomes, and a check that the lineages hold up.

The distances are computed with NumPy, because comparing 115 genomes to
one another one letter at a time in Python would take minutes: every pair
of genomes is 29,903 comparisons, and there are 6,555 pairs.

uv run tree.py             reads aligned.fasta, writes genomes.nwk
"""

import csv

import numpy as np
from Bio import Phylo, SeqIO
from Bio.Phylo.TreeConstruction import (
    DistanceMatrix,
    DistanceTreeConstructor,
)

REFERENCE = "NC_045512.2"


def load(path):
    names, rows = [], []
    for record in SeqIO.parse(path, "fasta"):
        names.append(record.id)
        rows.append(np.frombuffer(str(record.seq).upper().encode(),
                                  dtype="S1"))
    return names, np.array(rows)


def distances(letters):
    """The share of compared positions at which two genomes differ."""
    known = np.isin(letters, [b"A", b"C", b"G", b"T"])
    count = len(letters)
    matrix = np.zeros((count, count))
    for i in range(count):
        both = known[i] & known
        differs = (letters[i] != letters) & both
        shared = both.sum(axis=1)
        matrix[i] = np.where(shared > 0, differs.sum(axis=1) / shared, 0)
    return (matrix + matrix.T) / 2  # make it exactly symmetric


def main():
    names, letters = load("aligned.fasta")
    matrix = distances(letters)
    print(f"{len(names)} genomes, distances from the reference:")
    reference = names.index(REFERENCE)
    order = np.argsort(matrix[reference])
    for index in list(order[:2]) + list(order[-2:]):
        print(f"  {names[index]:12} {matrix[reference, index]:.5f}")

    lower = [[matrix[i, j] for j in range(i + 1)]
             for i in range(len(names))]
    tree = DistanceTreeConstructor().nj(DistanceMatrix(names, lower))
    tree.root_with_outgroup(REFERENCE)
    tree.ladderize()
    Phylo.write(tree, "genomes.nwk", "newick")

    with open("samples.tsv") as f:
        lineage = {row["accession"]: row["lineage"]
                   for row in csv.DictReader(f, delimiter="\t")}
    print("\ndoes each lineage form one branch of the tree?")
    for name in dict.fromkeys(lineage.values()):
        members = [tip for tip in tree.get_terminals()
                   if lineage.get(tip.name) == name]
        if len(members) < 2:
            continue
        ancestor = tree.common_ancestor(members)
        under = ancestor.get_terminals()
        extra = [tip.name for tip in under
                 if lineage.get(tip.name) != name]
        verdict = ("one branch" if not extra
                   else f"{len(extra)} others among it")
        print(f"  {name:10} {len(members):3} genomes  {verdict}")


if __name__ == "__main__":
    main()

## `report.py`

Write the project's findings as a report, from the files it produced.


In [ ]:
"""Write the project's findings as a report, from the files it produced.

uv run report.py           writes report.md
"""

import csv
import statistics
from collections import Counter, defaultdict

from Bio import SeqIO

WHO = {"B.1.1.7": "Alpha", "B.1.351": "Beta", "P.1": "Gamma",
       "B.1.617.2": "Delta", "BA.1": "Omicron", "BA.2": "Omicron",
       "BA.5.2.1": "Omicron", "XBB.1.5": "Omicron", "JN.1": "Omicron",
       "KP.3": "Omicron"}


def read(path):
    with open(path) as f:
        return list(csv.DictReader(f, delimiter="\t"))


def main():
    mutations = read("mutations.tsv")
    clock = read("clock.tsv")
    per_gene = read("per_gene.tsv")
    analyzed = {record.id
                for record in SeqIO.parse("clean.fasta", "fasta")}

    substitutions = Counter()
    deletions = Counter()
    for row in mutations:
        if row["sample"] == "-":
            deletions[row["accession"]] += 1
        else:
            substitutions[row["accession"]] += 1

    by_lineage = defaultdict(list)
    for row in clock:
        by_lineage[row["lineage"]].append(row["accession"])

    shared_spike = {}
    for lineage, accessions in by_lineage.items():
        counts = Counter(
            row["protein"] for row in mutations
            if row["gene"] == "S" and row["protein"]
            and row["protein"] != "del"
            and row["accession"] in set(accessions))
        threshold = max(1, int(0.8 * len(accessions)))
        shared = [change for change, carriers in counts.items()
                  if carriers >= threshold]
        shared_spike[lineage] = sorted(
            shared, key=lambda change: int(change[1:-1]))

    lines = ["# Tracking a virus as it evolves", "",
             f"{len(analyzed)} SARS-CoV-2 genomes from 13 Pango "
             "lineages,",
             "compared with the Wuhan-Hu-1 reference genome NC_045512.2.",
             "", "## Changes per lineage", "",
             "| Lineage | WHO name | Genomes | Substitutions "
             "| Deleted bases |",
             "|---|---|---|---|---|"]
    for lineage, accessions in by_lineage.items():
        subs = [substitutions[a] for a in accessions]
        dels = [deletions[a] for a in accessions]
        lines.append(
            f"| {lineage} | {WHO.get(lineage, '')} | {len(accessions)} | "
            f"{statistics.median(subs):.0f} | "
            f"{statistics.median(dels):.0f} |")

    lines += ["", "## Where the changes fall", "",
              "| Gene | Bases | Positions changed | Per thousand bases |",
              "|---|---|---|---|"]
    for row in per_gene[:6]:
        lines.append(f"| {row['gene']} | {row['length']} | "
                     f"{row['positions']} | {row['per_kb']} |")

    lines += ["", "## The spike changes each lineage shares", ""]
    for lineage, changes in shared_spike.items():
        if changes:
            lines.append(f"{lineage}: {', '.join(changes)}")
            lines.append("")

    with open("report.md", "w") as out:
        out.write("\n".join(lines) + "\n")
    print(f"report.md written, {len(lines)} lines")


if __name__ == "__main__":
    main()